# GC-LSTM-GhostNet - Step 7 sampled end-to-end (CPU only)

Validate the attached Parquet dataset, train a bounded two-epoch sample, create real explainability and benchmark artifacts, and verify the complete report contract.


In [ ]:
from pathlib import Path
import base64
import io
import shutil
import subprocess
import sys
import zipfile

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step7_sampled_end_to_end"
MOUNTED_DATA_CANDIDATES = [
    Path("/kaggle/input/cicddos2019-parquet"),
    Path("/kaggle/input/datasets/dungnguyen28101991/cicddos2019-parquet"),
]
PROJECT_ARCHIVE_B64 = "UEsDBBQAAAAIAEipDV3J/i8VXQ0AAHAfAAAJAAAAUkVBRE1FLm1kzVn7b9w2Ev5dfwWBokASSLt+JE6atgHc9aNGHMfwo73i7rDiStxdxhKpitLa27/+vuFQWnnj5tIDDndAH2uJHHJe33wz+kacTpLz65sPyenSuuZCNcIaMTmbJEdH9npvZ/e7KLpZaifmtshVLfCrWSqRWdPUtihULmpV1TZvs0ZjI35+UlmD1bVfl6tG8Rtp8igrpHN6rjPJi5fSKWHnfuVn16hkhfPubX03L+z9SFydJ4dX5ySH1ke5bWeFSppaVrnFaeqhUcb5k2qFv6pCZ7op1sK2DZ3hMlspf6/V7iiKrhtVObEHUbVtF0vxWmRtXStDO6DESufqbRQlIsOjWhb6D2j62+GHc9J8rhdtzSqQvNTfdDqXulnO2yL1V0yrWkLxTBbTGbQstFHp95DnmhqmamuVVLVyql5psxCXsv69hcq5xi1Xql57EaU0eq5cI1Y4P/fnkYT7pW6Uq2SmEifnCootVSlFJo01dJ7+o18KeU2tZ22DyztZVuwumTshs9o61x9c23uxgCEqx3eUMK1IectU5yn8WOsVds9rWwpn2zqDLTUW0UXD3xBCuyk0YCDbOhYpMhzpjTVTMJcSsKc2441S44aUpMhYmBLm9ldQlYSJFXlP1YnfAk2VkbW2Trze+daf/GbnW1rtXyfWwHelyrU0Am45PTy7ELqs2qa3x2DdmbMF3+oEd8L50AYnwRtbCz9o80E+4GzcF66iUysf8pnCjelJ3eg5fM2m886IhUP4NaJQ8k4uVPzELtLZ76AMqUtt4CqdDY3X+cYhkJXzkeYUnhiY+nFEkJORc/fa5Pae8lM2wijEkWAvs38S72RZuTi4OiYr+Yvy1YPwpLHQXIlCzlQBafJOGfY7pSkyGLZCQgo4RMlsGY7lcKuR7IiSaw6Is0vRWHEETbVhW+PJAgm7FCpfBJ2Uj3cvsdEl1qrKR7mXmhQWdg97jM0VTs29OWYyu8NJs7UwsuR0wLk/HyZ7rw6EMnlltWkEAGapHEdlSQnnKBM6K5JXWDRBB+7SuxLHN8vOfYRwtm7cCGLkRkkgh1JJIdew8+nkIhYEXzFElViNS8sGUknrWHhMEyVQsgjB0CGhqv3l6K9kJgtpSI3DXJa/cgxSqPi7ZBZhw8kOKV0k4va5ximQoKsqBK8uy5YzGLCU4O7ZMobfKOL48BnFOwRld95KrjuBsr0hdQEAdUMw0CgSKDNgo8zWMcVwpp3XCb9kUcRAKQTZ+F7pxZKscrIbi8PbSXL1cRJ7rGw9JpcSMfoQR0LUrSE/Y6Mqbb3mmxkSHqAYCcvXhPlrMkCC01fqcZ7ZiiwLKzuFt0pc74u2KixBW+vzS5mVrq3HEwKgnFwhEc7PODGAvJpR/TmKwZGay7ZoxHu5WMBsyCxAdgP8T9O0QWGJ8tYszKJdK7P3Zhc18bvdcaazPLeOKmRScarS8ig6fvCw2QOubHPtiypLZ6moCcuoWjdLPE+AqHU2olPFP2ChJKGfCQJNjO/8nrE2sEp4CUTEH49eU5GE0uPzVprkF/y7XUwTFPSunier3THLcGO+G8vlwhbqmxtT1RqtZVmE14hetb3m8xo33MEGcAlFoa8Vezsv37CNJl2sURlGFXalvVMeEz0mzCzicYDxT1eBP7UkYcjelGV+hUH/d4rvM/yMe0Ta2OGLyu3/vynXvetKSKHMAi7cPdh+QaUKKB6scUz1g0vlfU2sBpwjrJwyJLuRqf5IY5F6Q209JEq52cArOtI0+uSsSUfiBjULkCOQ+zYcMtgTEH7KCM97mL4FM7dlKet1J+xK3lMJk3nuS4kjqhkFQGlAI3IqiqiSTn1WUUZEo1UgtbnFXmPBccFhtVsKlGAOBfIFU0RCzdrXiz462KpUtiO246gLI5xZM7kCLNs670k6U1WqBbsH4zd9YUx8vc0VUS4wr4gYEy4mA6iiGrVlxVeQJJs4BZF7aAYcBbxqImysTaIeoGCg1OJlj3t9/Rok97Pm3gpfkdzzzyPc7xhV6y/G9Tigsxt/HSj/e9RcZNPCNeV0QWBpVDMl37/8inBnRQBf/OdMNtkycSgq4qDbnqsV6IrIqpYj3rdSXZOEn5PLW6aZIDLsDg7ZyfkZlV1VITD8+3QoK419C9RbuLC2El1hKxQRmMnt0SHxu1I/qHxYt0P/JGatyclNXJUiRKOaWXtHl3KSqL4nBaeXt9RFUATlIYTpankomFi8u7MTPDoCccnpolyJkwHDIE7p9smc+8msBXNrop9uJ++Pb/gRrjfXD+Ly6vjk7G94JO9dUqsFxd/V8enZx4uUiCSLfURcSJk5yGURbRJNHP56/ajkly0ibwZwbStADTM33/MNOcIKJYW0dJRdgQdcK4gB3yB7rX1bSTnLbqHC7ogzAXDyhEASCxYtyXJdLrzaIvdIHywIAojqiWfwv/fvE9kwwPtXU9769bD/V8P91VeE+1fHt3oAKSM0l4Ek+/gQLWIV5qhbbz24HJjV0XBJRM7iB2E1NTBAOIsGXc6xg+MLeUYr2RJ404TH+/D4xqI1wBIMnUJDeGAi6pktIzvvOg0KF+LiquhB2l9g0HqlfweL3YvF/j9TAYUA7euRmGziLtImK1rUMaKhJQxSjwMxx6+ri1OmzqIqWhdAOH7c/sUhT1UezdEcg8HGg26Eu5YRkdpQFltUFNm1FlDCzghAYgGN9HwtvE+oGnYNUKkaSfFBFBx4iuShlGfV75Tn2mjqYAKiIvRqW3QXwgd9BRtzjQxoDjd9MWIPpt2+/17MHvxJDF6phTLKTw98+VPc4NAcqEPMP717iW43sIEgvdOD71WhHI4bO950ip/fvVtT9/fIExYZWI+PQx6a0UjF35GcLi4ROs/2Aah5pZ/H4vLoJB4OWCbXv3B16H0QBSmA4V2a7BRipV3bz4BCo4/YPTmcfKRQxNGLTdLNwgiEWj/gH0hOtDU8YzrBYOfu0GPiJzVwyB24wi9FCHrUgwYrxZcYtGoUXMGewRpIOC5VczmraRSI21cF+vclTxhLJR0M0iHpzaBWVbVekVsDPvc1qyUu5suknzkyQaAGXFJW4sKQG6WPYg5s7qybkhBm0DhGpO8PT0/Pj6eHl2fTm4/vjy9oFY8nOVABbnJGHCj2iuEUmm+pohAMV75meuRhC4oXL6gmntHmFy8gpOd2I3Gqm5/bGbCL/nSR42LjywzhpGEuBp1aI1dSF57FQQLx5y0DkJ2++YbJ4Ou3PQMD26PCq8jVbWNLdvam5iAJ2CTTTtBIV2szSyH4nsKLaeS8hX4zS1bMh8WL4vztxtqAnzCK9F1l3I1PeEq5BX9bfc+Q9QKcQBJxyQ2/YX4Re7PikZxptAxAMW26GjLDf5eg6ndxKBI+4voMJINH121G529aDZjr9XSjUKD/W9HyH/fWLD74YgonTBtL/0vhrJ+owU3vVG1UkXRgHc5/1joebJHtg6PBuZ5zX8K36pYSCyRdnaYaBbJX10gvqmTBchtah727+zt7Owe7371On4tKGz/f7hekT9DpgnRe4d8OdztS7UfZneef2vkEEfd7UvTuiOPpomrfAgAKp6CFsZ6uxuLm8jZ+grQ+DzAQMqb7JCDS0QL51s7G3RM3RtAmwUTrskgpYUHQgIieZY/Ex0qZCEnJSSeSd+KqNb2VkRi9PcKrTjSyl+x/p5BiaSkfpo5CGTJ+3E0jmllw8nlUISiXYuAXOIS/LmRrcPWMgrRPqP2DnQQMEXwp6Bd9srOY+cime6ts4Yd+NDqz3YBM0kwR3NaTrCFq9+SCZpBo5FABbqyHKEGJSWziMfoApyaUAsVA3+9hC8owBBLu5lsMajwDz6H+Ti8WRM4ewJAIga6YeuVDzQOqDUZp2xDbPXh/fHVxfP7jX4nCCEx/ejiZHF9fY/tv07Mj/+T6eHJ1fDN44Z8eHZ8c3p7fTLmhiK73p9yB0C9uPDoGq/r2hn3qOWnfspPxIG8kfiUKlXJTMg1JTynu9n8ElKnQo/XBmtt7wwPKbtDa9V4ibPY+7aaY2Ftupu4hMnARQ824BXjtvx2Pf+i1eOd/sx7vGHrGP3B2JJQTOn839tMQMFLqj4JfAB4lzdTwTwgIH1fU6fjKqM0nHnajItn+uvB5OVN5zvOOLmMiZimBochyoJefSxAkMZGQ29HX8QU//Nt9CWRZc+lH+lI/Cr5RDFKmn6wOaH+LukHuIWuhJQ0tB5WuJOBUtOEqYW7yhltD2kbtIQ29e3qjfbEf5JSs6LOg74IJF/z3m8c0kgYcye9hzucJF1uXFKYhNw2JVNXPbPxnMWpUqM5lyqc583/vdjujr4OqI3/jfPAZpRvjoDXFPfvQ9OUwak2/daWWOoNYJL4t7AL4d0K13MNrwh/nUHaVLPmLlG+GBh9ztYsq6TPa8xoybZjxPJ48OBt6K1XohSY9eFLL33DIjlTf0bXlUUfE57p2vjULc3h/gY5rdIUFIvgjAcdU+EAb7hBpCgLypgyXHXrLf7zCxdMQM9NNdxdKcupthy6UyGOUJskwVtIxHrBpfBvsUlJSDj6iOsKHTJoQPKV23ZEU4NFQ2IAkj6J/AVBLAwQUAAAACABIqQ1dKIu3I0QAAABJAAAACAAAAHRyYWluLnB5SyvKz1UoLkrWKy5JLTCJLylKzMxTyMwtyC8qUcgFsrm4uDLTFOLj8xJzU+PjFWxtFZTi40ES8fFKVlwKQADiaGhyAQBQSwMEFAAAAAgASKkNXYEQ5TiiBQAANQ4AABAAAABhc3N1bXB0aW9ucy55YW1snVdNU9xGEL3nV/QtSdUuWbCNbVIcKOzCVDnOxuDkkEpNzUq90iQjjTwfC5tfn9cjrYBFToJPgKT+eq/7daNDSE0XjWvDyTdEczLlCf18Nr9avr+8ni8Wh3hIVLimcy238QS/ttFUyaWgKu9Sl9971sG1J3RdM3W6Y0+l40Cti1Ty2rRMhe5i8kzZJpDzFPhz4rZgWrnUltobDgfZmWk6XSDSRf9p53mDyHRj2tLdBCq8C8G0FYXOmnjPmlYpkl6vuYhUWB3wQFuNCL1bJKKThd/gki9YrY1lZUrqbAq0ts7574Y33t3Iix/o+eL18ffZWNvIvtXRbDic0O/RNByibjpV6U4FRkKunREereAUzsRBj8+MuC07Z9qoxEr1ZfyRveL7mODPATEtHGir9EjIFB1H+3RstDVltlVrD9h6uylGPHfOx0AvF6Tbkl4tCE8LQTZ6bVpBVADcJ+4uwEN23maggUdvTrvogTRoXhwcP8thFgcvj/bwP7yL7NbkEpCd9z765pgCfG1uuVQv1GA4oxb440luBnWX41NxXX58O9XkXS6mgFljcrMplJ2ifgTvxdnlBzKB+LbjNggaa7R2BOhr40McOgKNGNgCzYcQLndRxvfiSrDPjM35Fu+zwyGLOQpNTDWAxdfVHq4Zw7lr7ZYaLo1uaS/pPUwl96/D61EXmuDs0ITOg5n/EoUurawJdRYT3Zg224ooRM/QCox0fAjVOWquYCzIeo4olMu7vi1NiN6ge8cuHUF5EOF0cbA4RO8opGgaHZ0Pp4eLxew+dJ4bB5inAHvgS+kU3YxAb/8nchCqy6ciKixMtWCFjFS9hX2nvW4YmYQvwNp5tzEl0IGi6n4GZZIFaO2L2kQMqmgv8HWI35i/WeQ3RmC3p7mXY8dQ7rVAjd4CXwy7x4i7JjOA0mtg962YWW6Qr56AHnpT1OH0CPiudCxqFRD49OjF8YxqkUMAw2DkNUC0Xa17Js5K3QhNAyB7DDzEQ4UbZghs3+1qR8VTGXhzdn02xcAVkLJW+3fX18tHyLtVYL9BE/Y7A+Gk8YEOJnht2JakoYQkfYkGFSWMX9yNBqqMTejKcaZ3dFxnIRFvWWMKm0qEzDzkDWfwErywFn6xKrFqcjiw22js6WL0OzbqHkmld50Sh+rO4RTyg8RCctvUsDeFAg2BZ1jskSvns1jugn01FRcfz5bvJqfB665WGEAAmv5ty2EMCkhBBrjkeXRz+SmumtQiydza2Vs/JIMS5e+zCVJBRBO3WNxlBbVJFlVidHT5p8buKbaARgqOdfOYrOyZ+iRl5ErjMXyQlf4u6QUsdEgDqj/yZll7EbTVtl8dKdtenH/YI6u/HebWyc5oMQTwVMDu6t3ZHHM1Xhq5kIDPY00AcMhCPuyb9XJJ0dEbUcFBevEkVysQyvlC/YXD3VQvPDpzUPP+maOrynOF3hhf32Yf1m5VSB1ON7TShmtTwFN0nbOu2j75Knr7y1SzRMavHlY9Xl/oFAmTJ1ZHen91/ZPIaIFLTp4NDsYbNTw8jXbrazxhLbeVYO3zwJd7e/63nAaJ/GUlcBv2VndU5KV2F032C2DK3/B4XgUtInt/Kd5dUsdiE7hI/Zc9u16u5Mx9nwy9+nHMVAZCOiJqX3EUVelvFWA8Uk5Wr3hyAfZlqleqd6yez4bK1bOj3bNDyHtMrQgFtuJQaieiMBzwT5bnt79enk9eabv9r0remIIf8Zwg0XKZWVOYmFf752Rk3tCFkkA+KBwmHKdHS+fLT7NMDcoQwRRgMOZVxuZi+Wk+wH4/ZemZzgVt95Rgd5gIkwgIxSQ4hrgOAiQncsD4sJ/RTY1/RKgvgVZc641BH4EZ6BWuqYZ0PnKxdEuDwzc07i8IUt4oQ5j9k6dLSk6ZyRMmlVrptSzPHTBKYFLgyMvd87/Y+QdQSwMEFAAAAAgASKkNXdP/6pzZBAAA6QkAABIAAABwYXBlcl9hbGlnbm1lbnQubWSVVl1vGzcQfM+vWKCvOihOmsCFngQ5iQ0krhHbRYGiEHh3e3dEeOSFH7Jc6MdnljwpSeGm7YMsH7Xk7s7MDu8nulETe1JG93ZkG6n1qovPnh0oDsoO2wl/6UD4ittac9pG72y/rZXGx+GXz+mR47bViI1es91+GpTGevQKcTE/PDtUVfXdB+dfqKgCR4Se/0zNoLxqInsdom7CimpllG24JWVb0uPpcae8VjYGUp7J8+R8xOqB7gOjYiZXB/Y7LJ2fV40zabS0udpUFxfu9sXzs18ouOQbptAMPCp60HFwKVLnfKNtnyvJm4L0LMhste3Ye8mBmj9oW41qj+0ADPEHunZ+xP9/MXWsYvIcKDr64/ni7E/8+lZHclaw0JZ679IU8GweVxmegLQjOjK6VVE7u4wc4qkmz52OsWQppfAeEM11hCC/YG9iqfXd+uoaXx9ZmZKtkjRlWY+TYeE2JyFUUo7rFDJ1yay+3TFyC3xzkBCi0SiIDox2+W+FLMnhIR+KIBVCGqecQSr8NUWj2UttV8GZkvqt89LhCRdUloS+gk8BxvPodnxcOp6Sa/sf6e94hDQAhvMtRJVBPK3h7K7TDU6LUBy4Fo3drn9HzA0KFP1QoyZhc9lpw0vvHgAHyrIiwtWRSeTE1KD+msEkgznbIrLxXNo9/KjCd5trnKOmQUTE8cH5TwSOdNRcCmrcOCYLAvIWzwXDMOhJQL2NPNFLSgHRJW9lHMgi61osdd6NZNXIYVIyNreX6+rFq9d0m+W/vAAN2paTr25oUGGYszrQ3GrPjfDCbc8k9nCaqs64hxVyULKntegmZ1z/SDpANwCpTOR/YWdG7EDvb+8+CMQNB+koHgNQ/SeOS0lLgT8nBv7ftN9yp5KJeebOXgMyG7hJUYPAedBBXcgjRQH21DKdL8jyDj013pUhUnNsJTT3alrIOVH3yaVQqF5ARhQmo+OP+/rIAVRWr54XJWUNA4tB90NlkNQcTYJ4H/N4ZZVc6KBqg8CEtAb1jKwsKsNsUlR1MsqL1Yk0j0MsVGHP/IAD1ZQFKsBbF7cS7trUzL61MaiUjKrZCHrF8w28NtBv6+s3d4RJANJQfeTeeZ0xPs2C+Cqs80T4d4ZaDl3NoHoW0cmOUTiZSqJj3h9B975af3yf+wIyU+uAOJDhyAWlEYPRf+1daPY6PgqQbEPB8c2+MakVlYr6d2eQx7x92QgAujtO06Sno6M9gdad8CbKaHmnGwm6AwKll9YBG+zJfiB6UnTcrcHhvGXphUoAIUry2bVpc3NfLJb3WaNSBg7kPXTVQFkYZcHvc8L0iVuvaHN/sc7djnqP0iCARocj4e1RNP/qMkY9yg3f956/QVC8z84CvBs88+xGqHiHKzD/kncGmgwGAUYStTLL02h+PSCXw6HxuuZ2QTUur2LRaKaYVg7RFpaGqygKohtnRWzwIKbLs+Xli+XlywUZVt5iHLsot6xYGT0wpgc6FayKXSxEXlYUDzCfKqeDqxdHefISfwPEQbCqtREFHWh92podDD4BRTWiX1E+yJ1cwN6iJuhSGoQFlrcP0QlCosOLg+yfUo3BGjIxN0APQ1b8ta1yPzOOX+tdPNHDIlN0BScFKWKn77xq8XIV5SYUg4A/dPBwcV+5UUOqy6vUP0rhC1BLAwQUAAAACABIqQ1dwWaIt08AAABVAAAAEAAAAHJlcXVpcmVtZW50cy50eHQVyjEKgDAMBdD938XJE7gruDqmVTDaNqFJkd5eXR+vtKwdSmUng3aqVR6sfZuWGRb5Zh/SQbXgkpA4wKXG84t+mCOTaxL/Xa05JwRxGfECUEsDBBQAAAAIAEipDV2/f9bhXwQAAEQMAAAXAAAAdHJhY2VhYmlsaXR5X21hdHJpeC5jc3atVl1TIjkUfedX5HkrIoIglk+u1rpU7cxYY+3uY1fovkCGzsckaRB//ZwkDShi1er6onQnuefcj3PSjn420pEiHQpZcbd/5FLZOv0SQRrNA/lQGFfQSlakS+IeC43v9Af8O5WN83JF7F64nw0FVklfmhW5DRO6Yr5ckBLMOjOTNQ668rQSQXTtJkX1pyn29tUOl6rCB7JFv9M/599cRY4qVosp1SelqRulWUWBysiOrWVYMKGmct7IsGEzIevGfQhqyCfIL8jZhgWpsFsoyya3nk3umTUu+JhHMGCQcqtJLMWcWGbkP4I44jdGWeGIiTI0ot7WK+VkhSXHylpI9SJ2/F+0Fe3+8CiBa3QkzIQDeUQ6inXBvzUBAYMTUrOZE6l8nvW6F72UT6877mVkiYNOg85K1LLKQxAZeFvL4A/y2788hjrm945mgJ0709gUee5in+KMMA8yoCzL/4VxyR+CmNbEvIirmOaUDykLnHQ0Z+TzxLzo1NXHMAdDfttgA4hTC6QbRU6WCVrqmdQxywWeaqnnCdI6QttK8h5vDtBerR0DHfG768lXNkNC6HpuZFLZUloLhawXpJk2TMkU5nNAL/jEmzo1iP1hHM5l5BOja7SxCbVEJ+EcBtPyOZBj/kXqL+IxZfoMLOZa1tJCLaWJM/9JOZ6P+a1B4QLcY+pyU+dO2EWSA1LMQmAy2UPYnEa18dKgy3N/+lt3I1TNk1wL7JvrGL2rqtdQg875JSSPIPPGNH5nmh6TRJ7NDOSJaTLutfaOOEuiWHgYd3Tl46M66Ax7/MZRTGktdWXWnqVaSu2RDvOmcSWdOLOGs+54ZdEkyWa1HIF6P5Ez/nsj6wpXhIMSMa+kK2sgzYNio45e+thd5sQa7vuJHPr8H6gUDv9EzrSGcbr3pp1HYMSc8R7t0B5dyWbweTQG/F+4YLw7FAbUexRjeyZNeS5IC7619YwfQwwKr8ySPoQ9GvOJUk12TLKmXOBmjecT8DT+wCVULlNnMmSSYCufGGVYgDPsrsAfJdwm3UKvsYad0SW/DkbBFR8GDH5pRJVvGC+fcqIPf16f9IcjtopdeX4PPMfEYTP9gc4wRUFEJRwDu+jhcyTSYpoeQ5taQlsIv0ho37/eHerqGc7Vtrz7/I5UuV2Kaol3aDr3npqcnaEBcdRcY1HzWbyTM9d+4pjDtK8GbbmCaIdFOOm3N+VRpu/iAkn+RcJpzDcqAtMRIUSLM/leoUdrPO0tabea8JWpqD4ozvbdVRuw2B3pln51tY308vVrZqPO+Izf3P/N1sIpSDPXxUYXq9K06nKDUdXlAuktE5vdU2S0f4iZX+0eC/jJW4iXg9QWiCiC3DlRSazDlIGKD0omI/Ug0vcv8FAbfJlpMZVwhk0EbTcW+41vQl3wG4rjU0cZLEhUNTyAQR8huR6u9ZV8St0EUvsT+SNskb+93+goQo8hgjlpimmwfAYfe0btXWTns9vRz90USyry/v+O9gtQSwMEFAAAAAgASKkNXcCH+rXdBAAAWQoAABEAAABjb25maWdzL2Jhc2UueWFtbHVWUW/jNgx+z68w7jnpbCdxE78NLXYr0A0HtNsehkGQJdrWKkueJKeX/fqRshM7612BIg1JUdTH7yPbO/s3iFCuksTwDsrkeeBm8zv+fn7YPL+8/rL53FoffoWweXh6eHy0L3maHTenDA94AFkmuxz/7KzEsxX3sFpJHjjle+NNo4HRVw+hTORgGtMMZzD5IcMkx+wHoYSU1seUPXf/DBDw4HSCSeVYqwweXcSxKY71Djy4E0i6nhtVgw/MDxWeKq8Gj87LgYuP0qNZ8wo0E9xIhRbwZfLnM5nWo2edPGju/ToR40fgroGwTl7j51+YIKgOb+Bdf5vk9WJezxEU3jt7AsONACasHjpDwQzP9mFwBNN5nTDm7eAwolaInJILi7PvaKBESoIJqlbgFok+/Waof9iP9NM6+Unb9+TpcZ28xMPJ05d18oilKMODsiZ+f10WZ6xhZujAKcFq4LGiOfmL6pTW3P38+vqFoj2e0uBZjyVQpWWSp7tDZAQiTS9snB16qhmP79JjcQMXNsQDG5NMMfsUfzAIvvbIRpBLHNCdHZY+61SDL9FzhYdb/xD6IczeY3aTWbTQcdZy32Li7T7bbg9Zsa+zPD/u82K3q4o9FDsOotgfQBa7/f02T6t7KXdFVR95Doc8K6pCHO95vlr5Xqvgie94LeIRHFeG1Y4LQprQS+/u03WS3h1Swu7ENVEFfdcgZmu2OIwtvMsIjCutGA8Buj4QEnnEGQORAmd0TFBrOIEuk+AGQP8E/0CtWTIKEyIsyJ5msIOPnKq0FW+rFaoJCSrAe2Uaeg2GncAFpkytjApnFizr1Oi+XCOd7ZkcEAFBVV4BX7qp7vO3XXgFEsKED17lrZ4gsqjyOJ2wg4ZXmhg+RcUakVATqQm2NIt2w4jrHQ/WEWSRWUnioEMFIuq2m7pkjT5f02Gejn8dr7pIwHHTQGwhdjC7iw3Ea7Xqy6Tm2kesqWVjgb0VrScxxK8VD6JlXv1L+tgX0UbjDLMGoGqPYxzXfctjmXejQQN3BnG+BqbTu1olUfusG3RQCDrgMMOikIMB+m251J8G0wTid7G0Im0UzWmikEOjiio3HsQQFEIzj5q5F+Pcm7iEozCwUcjQU+bxABjZW3VtZDmPHWr1PHc+Bt4OJZLpJYL0yWik+Z4LvLoRG+1Dt2loHRlcR8vdEdfRxM75kt4iMbHBkWzvykj7HrXB+5Z2C4iRNuOfcY+MM0lGYUQNLWfcbgR69z2tl1Hj6Kxx+iLvK5CS6pGqK5Nid7166uL/zZqfgei6pd2EL72JixM0Wi9h9OKIBS5CpLpDjbXcGND+kjU6aU5Y5N+kOaybyowMrdQVBJyllHwmdVx67B1U04aoeFwQ0d6CeBvR7QDZJGjVCmdZnS3g8519Q1Zc5PDBs1RGrBWJCbxjvh3qGsdUhR8I74W3NOCzYz7HAQ5R9nEJ7TOKGZvyISS+aBTiqh6iumMkdYjmCDWxjHQ9KeKb6IfVLOlxhizrHi+zPaoBv6MSf5S8+2P1XfWOWDIJgp9HazRj76UCUgQOFWas60ZN41sRa4mqw9TCIrGnGcW+kX/K1amvCDFOcqF8XCuGiUHyuasSkLWYQ6HoBOO6wU0a2m4W+6K9vCaGX4tgRP15KND/WjQ9p7BpSES0pqD/AFBLAwQUAAAACABIqQ1diAG9gdUAAACHAQAAGwAAAGNvbmZpZ3MvcGFwZXJfZmFpdGhmdWwueWFtbF2QUW7DMAxD/32KHGHAsJ9cxtBsOtGWyIakoO3tZwftivZPMCk90k3rD5LPYZr2mjFPjRo0FmJfy7GF0BRNa4IZy3La+ByjuZJjuc3TQiwhFJAfiohrF5JzleE2ukYIfW/I8+R6oL8pTOBfH0+h0GYvink/Zo80LAWqyLFB8mD3NRqEJ9Ww4R+aCqX6hg2LUluH+h5Heu+YUVj4PDCtZGuH9SSXqr/9jrPfxmJeXo2p7vshnM4s8dK/jCU67+iz5HoZ/R9F7snN0eJnPMPEnYQLzEP4A1BLAwQUAAAACABIqQ1dNJUNTrEAAABJAQAAHwAAAGNvbmZpZ3MvcHJhY3RpY2FsX2Jhc2VsaW5lLnlhbWx1j8FOBSEMRfd8BZ9gom74GdIHd8YappC2zzz/XoboLDTuGs69p3Rof0fxFGI8ekWKQ6k4F2r5RobGghCGYmgvMGPZV5TXmM2VHPtnigcqk4SwgfyuyHj4EnU580aPDKFbQ01xo2aYjwoT+OvTb3I55npcirJR6X+iu9J4O/EFXO+n3Hw6LMXKOh2os1lHZ/G8KvmDGlc6Acs8AyM/hzC/zPJ94n/ClX3JRseYOP9UwhdQSwMEFAAAAAgASKkNXer/t2JFAAAARQAAAA8AAABzcmMvX19pbml0X18ucHlTUlJyd9b1CQ7x1XXPyC8u8UstUXD2dNZ1cckPNjIwtFQoSi0oyk8pTc5MykkFcopTE4uSMxQKMgtSczLzUvWUlJS4uABQSwMEFAAAAAgASKkNXdO/1PCCAwAApQgAABAAAABzcmMvYmVuY2htYXJrLnB5jVbbbts4EH33VxB6ogKHlYM10BrwAm0f8tItCmzfgoCgpZHFWryEpOJ6L/++Q10sxnU3FZBE4pw5M5w5HKZ2RhHO6y50DjgnUlnjAhFamyCCNNovFuNa6Z+n12/e6OndR5wPsvTTSpAKFnUktiI0rdxNrF/wczCEk5V6P62/16dzFOu7INszlXFlsxh8mDIVtJPP/cdPf379474xPnyGsCT3TtjmgwgRvqigJjvQZaOEO/Dejy4IPv3r5tK5N+2i7ybhGZbLBsqDNVIHHnezwf068k+/lQFgumC7wCvpfrQdhVOd9RuC7mRL1sWwrEB4LLcCHRIbGnNy+zupZBkekGkZ6/K4SYIgLHLTOWSeWJk64Aq1wkXe7VfXwZLAd+wNN4f+M59rwErb0ZzBs2hpPu8fI/R/WTA0Q0g22I4yNEMvmNQ1YIAS+rLSfMgvPrVxBAWkiRN6DxS3Rcf95wnqnAHtA+WpO8Irrjwm8fD4f7Rp/S65UY4uQIUcUYbMgqt5aTodwHHtaf56Ii+TYcJa0BWl1+nI7RQxJ2/IihdFEX9YMXbGVVitmI03EURn4gGAhCVuQ7aAmFaoXSXI02bye1BS0xY0Hb9juNUyKoY6TKKiT+SG/ADI8Xm8EO8knQs5D0k48F0bIX+fi5BV8CxLyDakl8FyNvTV4l7+FY0xkxh+FI1wewj8lOcJftTACJ4UkQDSbo6oFw1OoFP1WhFQgiesIrfrAp3mMtKCrYvXfN6tL33erV/xwZR0tkmmHavjUtrQhCA02J99E0+pF8q2gEFRMR5KoyukuVIz7OSqVw7q6Bej9PLtW8HVDlnn5rLIgOcbz340IyVdFXe/kZsbcpcyWBAH7rwf/IfZy744U4JHcTMFyrgTxzNv8AtxPyVCOTmwgyNmiWSfjYYXgH2crFiCp24YH1dB80Zf72zgAS+p9pewVsT6r+74GtGlUTglJV5iUd9408VRCmVoT6NJ7FpINgQVVlHvce2cPVqCKU07nox/+9/jYMYiZefbh8W7MsvZ0ckAPMD3QOMKqzplPR1OXjzRFSa+xZISdDQVRttmXahv36YT+Bo/d+LI8GrGEAYHFc2O2ZJoOLZSwzbLrvAR4UkjdNXCPDn77BxOAGQaUnV0wOQXmNFqjvQhPbkYKJt7lz3+1M1T0J0ChycrUfVymKJbnF3jRML/R/Q4mBb/AVBLAwQUAAAACABIqQ1dzcB3X4cGAAC0EwAADQAAAHNyYy9jb25maWcucHmVWG1v2zYQ/u5fwWlfJMDR0i5dC68eUHQbMGzdCqwdMASGwEgnma1EaiSVxMvy33c86t1ykvqLLJL3fs/dUblWFUuSvLGNhiRhoqqVtoxLqSy3QkmzWrVrqaoP3f89N/tSXHWvn4ySq9yxqrl1Gx2f9/jqN+yhFrLo1t/Iw2q1yiBnGUCdVKALCK+4gQ3LRGovjdVrd2i3ZuoatBbZ0U7Ezn6YLW1WDH8aTFNatiWFY8ff/SHuER3IlWaf4bBm17xsgAnZy4iFhcqEkWfkfiJnwghpLJcphESwJqkR+igb73mxcQE2ROZRe2rgNKh2ifs71G9k+mijVSvqCaFEtzzAZmrmiFgDBlW2p1t3l4pnSapkLgrySOICtmHoQvYfRWvNKpUdL+PjdyUBxbnHSd9bfRj5zsf6wKuS1uA2hdqyX2j5J60xDNy41Q1jX7Na86LiGyYVWoTxYGfooBpkBjI9oKPxABiQlinJfuVFUcI3tVafILWMYlCWvWDNhYGxnDB4f/j7zbvfHBsN/zRCQ8asIm+wjov3SqMp7YOIUdqidsTW+QqNd7bEhueQONLQeWZwYxRrQO9auLUhKq0yTPht0Nj87FUQRQzNvbtftUnVO9mphFgjtw6u6xLyhMie+skiRzbMEDcgzGcNJpDIuIVxlkzyyS202TQ/6x+LSB3s6wOwZZdB6/xgzQJkxN3T1KWwxv3DiON+CgZRVgQ7oq4EvTlizP8Oyw7EPV90r1tyXsVlr9Oud7yn38yS5S8HG58refCuleFJmUEFXSXcsLuW+j5oK4nmfotg6A5fdurvLgPVWNCJRQEy6U8GvSYGsxOyMMew2vA2IlNuncb92Yh9hWaexy/X7Dx+tXtA52VZrGoM5bXFZUxlXC4PyOrlORUv5HkeTIKORD01muRVOzZs4XSi8mSkQ7CLOjNdGM7jc/Z6Uchr9iw+f8iwx2V5K6+okIfn62dRa1OhVVMnWt246Ag5GEKJhmYYTBgEDSTDyZHeI/LXW/agjguMeqVqZYQV1xAMvScXUGakbc8zKPkVlEmKYSFAYfoPe1ZUgAWuqk/sI0auQbomhDgsm0pOdhHY0gqUqZd2JbpTNlgNRJrkwGkKmB4btTDqLeNc956knkdGrdnlbmhcbfDn/dOsWSmMperE5SFcOrN2rcdDom/SnnjeUY/R65SK70if+z4MnGQylTNv3ZnkFTghiGbTxgbf0LXFYWTgrAKRpUFbBJLufNDnTM9hVNQJav1GW5TuggoywaWrcoXDy/1D+TUX2FvlmThHEpNej8BYqL8NhvI3cKedcbmik7vJwINYoeURREqQhd0jPBCxzx8JgecZz2iHUFhWAsf/z4PoMbEuQBmQ2AkGnyTXE59CYh8yR0NxtRx7ok10U+IZV3oRlcYmHn5QB4+J/0NidfX8Rqy2EyYuM9JGa8SkO9zUNXWBkzp1occ5qFbooaRWpUgPrX6ZVnVyI2Smbr5EuxNMtyN2X6pnoXm9TzLsv2k7PpF+9I5kX6DcjNO24/G4RtPaGhjVaMyC3kgPfJo0MBRC+qaysN0vuVtO4gqFqXmKOTG14ri8Df4gNaK2iqFa7qDPbdrZxS4363DG8URB82yPKxrW7jOoanto69gM/xcn8X8xw//FCP8zJ+IIcJNAdQWZGy0xLBWVLIrRXmTYW6ZrJT+ApuGtNLaaHDky1f/8wYGu2CvES61FxfUhSfd4FcUb0LBD8/lCLLrycdG6eKlkPODgiyMHL9eLfpzpxiMvk6CIQ4mvkJOZZjnjvcyWajzCXI5GmFbmRNDSpOekdp2lGxfvn6bBErteHRoVMSFGk2Krkicm6Kd7SD97wFSAiZi24K94qlWSP3s6+C/iI17bjouD/xz0LtPbLxdkgMtQN+06MxaTv9sc5f9J+klL7Fa9xRlci7TrEWndPGrhhz3dX9vbqmFv3388U2j19+yU+NjL6CPhpEwCcKVUGU7VqsQtZIgcSLEmuTFZJmmTcbwIPqbf248/vmFEznryXnQmDL8qccuVBruHXvvOomOALCiXAeZZhUZi6U0TXhZKC7uvzBO0+9P1zRf0HQNntuFC0N34DJswZ2Pm7UXVR5sK+kOXVHzzyhgciFHOv3RLdV+34qypatPSrunuluAl02w/aJpWoeZYmJQ22zBYu2K1CbD8gzQuutykQmx/5uXsJt1+RYvNnj9/8V04CI3pLg9hf5OP93CbiQL7Vhit/gdQSwMEFAAAAAgASKkNXbJmmGeyEgAAJEgAAAsAAABzcmMvZGF0YS5wee0ca3PbxvG7fsUVnc4ACQhJrpNx2TBTV7YznrqKJ07zheVgIOJIIQIBGAfYYlT99+7uvQGQUt2000c0iQXc7e3t7e37Dtq09Y6l6abv+panKSt2Td12LKuqusu6oq7EyYlua7dN1gqu368zcV0WV/r1R1FX+nmXddcnG0S9rsuSrwmRxn1R91XHW9mfZ122LjMhuOk3TRKiAVwwje59a1B3+6aotrr9ebWP2WvAm12VXD11dRuzd/x9z6s1N+uo+l2zZ5lgVaObmqzKoQH+a/KTk67dz08Y/Ojefda29ccEVg+oOgJ7f8Jv17zp2GuCeQkA7ZyxX7Omzba7bM6qGtb+gbdsxvgtb9eF4Dm72rM/ZdttyU8LYMG2JQ4zXn0o2rra8aqjaZv3bMEu6wpIpoUm67raFGal8i1F9sesrLM8lS0nJydvnv/x5Zv04vnli9cvnn//8h3gCYM32RUvg5gFpX64QO7iw1o/dLC5vMOn7+VTdPL2u29/eHn5/PLiZXrx7Zu//PlSYkvTddaQsOTZHgekqaj7ds3TTVHytMi9NmAbNkVAW843bJ1VdVWssxJILvtdlVbZjof4z5yJro3Y7Gv8LbnfcpimwneCiBJ4KppwhKv4iSt0IlS/52bXlzBoRXjLQnT0JrGb4bCq5TRd8jlim7pl8pkVlXoSK4kFZVkACiXUocEUqf4S5iXpXzABu8dzWgzhxIdYopCIEVdSdHwnwogVG9X1NTuXyKhF45OrID5lIFvsh6zsOYlhuAkuiMYZzeSQkG2ARPbxGqYQTbbmIKTtDhlIcjhndxb2PojcTTDLmmL+pkVu0b9zUKDkBSjwK3wjvrsNY9ZP7iJulcSXqKZIUyP6soNhurPZh26PBncRu8uQUGoNOe/AMqV9VYCoqLkPCFCM6PICLBMfdzV929RiSoLdhQrehcdEVq1wA1ueK5GUE0rxM29F5dBCIjHokjOstMSUvAoJacR+tWDnR8Xm5W0DHAE7xW+zdVfuGdggdqfWd691gGzSHe2RpSS6jxXtd/RrID/UtjxbKdaLDq10KrJdQzZDHJOed7wtuJL3XSEEGn1kkKLGU87wceZIqRIOAj+HA2l+wzQ1zXEdIx/J1i1H3pul/N7Q2LRg/6sMBIUZqbpTnZo9TbZHC46W1cxFpCzHC1klmQCfx8MADWG1DaIEusoqC4OvFNqvFVr8+ZwF88B5G6JVvLBYX1fdl08B6aNn8XZYrSTZZU1YZrurPGMfkF1zHSck4jp78sWXIbUmoD91DpP03Wb2LIii5Jrf5sWWg1BF0UBK1td8l5G/O6ieOZI8afaNMlpWY6SS5BAFKEvzE3gVhVhjiqIY3LJAN5eJdVEsXmWlAGMtOAQBGFeIRRjEKFvzIPL4MFitZsux9cJy/2CCHtCE+ideLb5vex6dUBNDlQD7gaGP0oO2rru5jIXwFUeng7ZdVhUbwD9sV1EMiRVwrOtBbpfYG7MkSbSGpka101zOTnhEKHjWrq8dpDG7hmDGWj/ysxIrtMcEs7JO11jRaTjU7JVPP3mxBQs0HaLf7bJ2n+A2Bspotky1oi47JCbttqyvQg9XZNUaVF1jA7ZA/JXQZGAoAz1EBBZcevSqK6qeW9MA06B99/HIXwbIbBBA0q9TuR6roGJdt7jMM9NS1h/BWy8oAMIxUUItYeSSj7zX7bh2evQplpg/B+N/duYONTQl/Bb2AmKOA+O+8IaFmn7LIXwDtcadhdghbTnGqXJ/ooeQP7HIrXQkWdPwKg9DAouJZb6aqVjKDonZDd8vlOXBGGrOQvwFTicmJ0gv5yvcmY70u+UQnguuNE3FBIWgqF0LfUhMyouW5Jv9TUm8FkXoSI30g8VQ/lPrFvTA/p0/I70YK7EjpwCGHWY6E/Chi3HlecxN6ZpewYyXdfcK/az0UM4og81tHG/hpIJNbaAWeYuMujiYyPnEbgLoY8zJmK2epJOztfbDk6ZDPPCA8GcTXNa4V/0aU5hZAwEhbz+gt1Zzs49Fd21sjzid4gmDNKr+yO4c2u8Db6poyCpLOMgjSKFvtSeNgme/HQi7Z2NzDmBkUUOlHlbFpR0MPtNJLLggIxWoHR4aChWB/6Ev0COh+w7Sk2LHJ3jtxJJ3PpJ7EHSZRss4hEnCwXTdGWptLDkmzeG0ZxBc9QqtOCnJ8pgZ+0zTug+BXJ7i/oZYcpDOjTQ3L9aUOcZYZFh5ySmFEujgBQ1KCEfHb4F16PBBsBbG5atplKG0imA2lFCA2XCXolYhM/z5gJSD1GlnuBgsSiS+VB1UesJiTfpjEB1yAJIgiuEeg0Yl4HKAi6LYOAQlW7DNEKFmXS8CEtegwSJSHjCIBYZwHAU0pWxaAp8dleRNoHYAnT36COSAwwzQZLJGOCOE9LZHR/VA+/oGlfHOTBO4wbykBB0U6Jhi/HICYgVeCkEcEVgGRp0ceAGgUWwnq9tiW1S2nGEm9JSUZic+Lw+MOE7AYAwSYXXTpabvmr77B2iZgH+AEnfEQTqcRALnx7hKT+l24VTYd4DrHqRCfy/D0Kwowa…10873 tokens truncated…aldW3qgWmMhucwZi6zMLFNy7NJ/jK2uX/nV4zxarWm3GasTPtP9i3llqJxjRYD6od/DL3jrLxJqtbLAIeLypbr2SkpF9Yl4HNVfiEK8FaJdLlRKxuCDB52NcJevJG2K8+f9wlYroDvE6z2/u0NQ/38CNFFZx9+Xs7WVk0RCxQ2e6TYMYl91yfFcGx7jKhztjPX4XHuIHuo0OCuYhP3Vt+6gq0s+zPpepmLtkXhfFU20Elr425w84PgMLz/Ok72RNVhwTYjYtz1Z+rIuRxBxYGq98VoOVazATho0JYaNlZeMpTMm/mjWMYBUgfNZB5zH3+aHh0Ebs/w6OK0aS6djxwK0oyS81IWkGI66HpllcrcMKn/rvkxrTqymSsoxPNiZ0s15rvzYJjVZ+Zkj/edDUl7CSSsRLlxhX7s3CxiZNHvclabNB10c4EngfPljFbptH8HSH1pBL7fL8eVtbwd/acoIin8hWpm8vgBrFHNuO94cz42+IEjjkP09E9ugrraEGr+rC/wFKvQ36NfOGNRE0q8vBjCtYM3SqftjhWines4g4/bqGbKmubFLDhxyiHoYXkS3W6doMF44NMkF+9yh+jptcV/AarznSPZF0xHuYzbxfG0MY60c/o0Y2jl4EGWHjmGT2w2AiPmt6OnJR02fb6DMS0dmzrTJzGZMdg96O0lK5HVeLxOXXKJ1dbUu+PqJgk+KNVk/T2ZOWkxqHD9kTxzQyALr3rOZUyWxkxszjq3HDmlV5UFevwNURLJDYG1u7FB/1K3nLxj2M5KaULz8L7aXJ8fERbfJ3Bjsn2cXQnGGthPgpnmfWliS+t8oRvf6q6o9XRn1aZqprTAnGqtoY56x4ATi4ccRwu/UYXOjRAvjoeqn8SccStuR8jlLurT/sBt6jctRBSiZx+1m/mViOA4O9dYELwhGoiyEo89DYQxsEzgLxvLfcAE76ndfMcolNKRE71HEXC8/woBfZ44eQPxov/R+9r2YmGS71n3FtfMt1AtYtTgyrbpWZm42rKjmViLWP1hE7R44XG+MfSphr6I/Ol8wlBdzAnLWjUY50boECPcyMGVR9K06EiyBPVo/iRnNS+mUmfU/gzrYK0wzXjNAZlWPpt9oHFOTcl+DxcbiPOIHyWixTGZHBHGFPTMNbBLl6WsQ56zZMFPjCWP6OC9+j5CIDi3snPxZ53EjAhNi/wB0PWA/tBtv7jAc1fqXF+VdX8RRpTVGZ/l8fCffaJLekP9xLROSW0Bs9JNs37VdMtpPLK7o0m8zIQBzjjFacpFp3O/v13XmRf0v75mW7buXpZunezxlJvLGUvdcucBojAjmGsuTyF4gwHZzWLnAPZibrFX9pgSJQbu5d2CVxHS8Ghtzmqqhcf1KPlaU4v6GlPAO8iYwnhv4yImRV8c7kVozD9922lJ6rSP4sa/VfcL/8HUEsDBBQAAAAIAEipDV0ZneKhHg0AAFAqAAAKAAAAc3JjL3Zpei5web1aWXPjuBF+969g4YmcgTnWXLurCbdqMtl5SrKuzeZJpbAgEpK4pkiGIG1pHf/3dDcOnrI12ar4wQJxfOgbjSa3dXnw4njbNm0t49jLDlVZN54oirIRTVYW6urK9CXq3jZ3iW39psrCth9EXWTFTl1tEbQSzT7PNhbxFh71QHOqYJbt/1ycuPdF5LnY5NLtdRBNlZcNrL+66tphq6TPPu92LJhODKsTtjyhvCpv7HjRHqoT9hWV7apEkUIHzks1Reoul0B7eJBNnSXKCeFe1mIn46qWSaZAFrFKylpyr+uABpAeJ219D/11mdimaJMhNqypYFwq1WMemJZ5vMkKUWe/A/dXqdx6sYKN/W22A40skZfwK7W5t63FAbvS8C+iEV/xiXtl21RtsyQBc6+gGaqpA+/6Ry/PVLOCh/XyyoM/PTU83KVZ7VeilkWjol/rFlDkEabG5R09BjRbUxA22W7fxLk4wWp/MIJ0QtPXsN4bb8secf+nsCp2jHtplUXvbm64t9mUxzgrkr1UESM8dilQumXPrUcRhHXZFqn/MQibMgYjnYGBXoDJilQeo68iV4ZDlG2Sl8pKW/fukjAp81wmlt1agnMU3mrE3ojIwWZro0vSPShcm4Xy67J0qiqt1uZUtYfHsj550UDbPrpbmJciVT5hAZPMTA1xjAVhLUUaN/LY+LJIyhQ2j1jbbK+/Z0Fg+CkfFACv1lqGZe1ljTyAeLw/CK9Jt6CqyrMGUX3W1CIrQEbsXuRZSnGF9WZbokJRVRJU+chkVSZ7tiTCVuZpDesJE/rpl3uvXtEEelo/9WxiLDdEdyYH3nmUKAHUv2o3GDaUv+DeO47DClwx8hfvuffeCAy5EeAg3NMRApn6Pat8hOHAHpiQQu5EAmoWyQnbB5HUZbxdzIsFkER9J2sST08+JQsQsCcmZHqAgX9AtJINcEDMrrQbaHlHkd5hPViA1IfIp6+XhiRSboBWmq21pSrSP1zHp4jwgqsBFqzyj3qY/UTq4d5JP2sw7jVZk0vzBGZT5SKB6B0jRx4YEg37wQh3V2epL/JqL6Lw7Yfgk+7N5Q7tYuCOLkyauEgOBdgjl4NIYVyxhvC9l2mby//FDf+ga5w3zc7YV0dn6mQpR7QOs8e6z1ktGmnmD/tm1j0NzH5s9M7cvyNrR3FrM9EmZaxEPwy2cqaCJnul1fecUbC/mtUeEWqtw3VfY7dnFcQ0JSNjuEj7nZad5pOy2LZ0XEO2AMb4LXG4Fg9aYaRcPF2s8keox/4hE8MBEt1ogouyPoA7/y5TAAK4MM0ABH5Ve/DRuqNF4NwDDstFwMno7fptlstRuC6IYQgQghO8iSJjikBRD0AR/Ofe38tCUmiZTOroY7xHrFnSD14vmNBBHNGMclkge8Gr8P2HAM3k6H8c9AY9n88OkGABIig7O6h9+eAjV3iQU+LmA0ByEFXE/py3UhkCI/wHFgI740ld1htR+4SE9EXi2I8paJTxscmSO9C6KHbSR1JoF1jbHgoVBFqW9hnzOJ37Rj/cTKBOs1CkdQeknz6NXeK2lmmWNCRo6xafk6YVufOHgvKZl2OlNorXUd8PaOtaIpFEAAqPnAJBB85Dq41zQFK6EZsMIvzpm5OUU9xAtgjqKyoKjc439EBYVCf05FOMm8xOwwE9jQBB1XQ3mA+3Ols2cy4Ousg9+Q82eu4To9coyMJl6ltMzCYUyu1OnqJcHDapoBTEJCKLdaCdkDJ2PBWGCbyv+QabzYVSkKwCw6KzFKLEWn+2NSih2otKAjYe3YvO3dweAKItM1aNSO58f+Fdm9HVkns3cDT0n1ySl8SY9eBtJXYZH3dhJIM4XiboMPh71qm/597HQPsaKIumV/ULs7u0Es1Q2x8KW4JPS4zzRhIdryAMmuv9GE2kAjgdezRrHR6ywg9QXtMRCDejZAkiXpMVreycBxlp8F+MIdle2vwxGDeW2/V0DuiugBAt6A6oweZvht+EbBWH3r1aaVuFOL/RZzu2bPpp+QjWPbJm1/Jzd9lLKJtubDl2YD0KtD2ZJMIJWkc7e0Xy/Ee4H9Mwkr8M322fAtYPtUCDhphsNcZi/RTdhFHuQeukT0VND2c/F/L6Xl1DfGy8X37+wjj7evsL/P8V/gfcpy2Hs2j4F9ofGreWgEEybiK8Dt52b03g0QX50/lsxhh/Fx28P0XexyVOfzbn1b47uuJYhzfnWLRiFIeAeJA0/AdZs7U9FJize0iRvdc9aNTXANkY1RxwbcXj9APtia31dgWsUV5eSehCuNiUX/5PyVkvySAw0hVGSdzEvxg0sGnDarE0XtxUGPjTTOwo5fkEt6wKyz2RzfoW0FfZZMD13vQuCdHoiqDlvdQ09qW9bKo3sBmEvewAGA6VQxpptTOaY6ihGabNlqbRv0OvGNxe19HbVzr9d3uaZw3eBx7Ne92fxxfyevF2cBeJzqeR33P0CM0tpIwfIJH8gKfQcXSimhkw8oA+5SJBud2CW3I4xXn/nn39MDRUvG1Dyt1ZscIOmASMc/Yv9HT0REwvj68NphYNIK/5A9f+DQ/9K5DNNo/cqKufTn5yeWSegcr5QvdYZ3fDOuNjt7K+Jt2byoNiZ25B+hKEPsYmDtVdhKg3Bfuus02LJF3ua2j82S4a52VvWN0WsR68JCX7pPOxSK8wgSSmPrZ21amod9nR9ZKkbItGoRqHK/vMsLVN4JZedwrq9XQWEsi6ywHp2R5qRIPZx5xnM65IgbCLg7oexV08pOUQ7ihh4hfYuI76zsRHXM8VibTVDkpE2m5ZA6eWtVt3uugKTzQpFEW6rLPurvymFmR4WXP3rHniw9rQpDRk7VqBL0msie3MLR69Ia7Aa0EzdLFlRzb1iXMe8IWsf6Dml11gxjasD1BJA8AP314FmjH+b6gAfZqP7Kb483ztRw/FSoLxpwpnhztIO4bd8UbCShkne5ncVWUGSgvmsLrxKWBvTBRp3FbIrZs2A9crL5219u/42dJSr2134V1hyRiZEcyzEFOmHI5yON2kN5ozNmN7MxXIXo6n61oeCMfr0Dw0J8ZHpS97xf+Hld6LZtvZpjNXlR2qXGIaEuMxpMq2TuTEaHvvfSBLgxvWDnRjHiFQtOdfCTmbdNmO3iII8Woc02Ll0w8XKgExoVHrdyiQ2OzLh0LHlnAPAP67mwvs4QeO8e89xT9CgPgHkjY2AmzudffKcrIOhWpOlfSRidVyeb2A6EQziLC17hqHjkktZVpKeUYZumiilSCaRhYYRWIdKC9QwvHbJQ45YXZfmlAdUjtu8K2or19cMSXIFuiJuUPoyLWSIvYg6dUYF7vdti2SiB2kgNNhm8E9lOZEl2vnHWmHiNCVPNQO1diirlZHw/1inVCVTJqIibYB79WVu/sMHCpTbL5gp+t1Iyd7VnPc3a+sgw3k8k2K3UCY3mOc0Iq9+EDAUtvMaeDgLjsPIG/EHAJi6oOo0zgHay+SU3xQcfXhBm9tMwM/fJgfIFWfzVoemU4F2RL35IysAaI+8rG60yfOHYZ1HIaY/kL60gV0VKS2V/PWRz8Q/iWn/9suI/atOv8G9prZI4ebsPvl9p9eChZRng7gip3iLoiq3bK4v0xbAAR+KknFcAEgLULWUJPvev/pLKFqJ/1kFnjXWpG/fy5O9lIKIBHOcHDAImDovg6M7oBl2iZwBXx8glh6l1UVtQnmt3Kjlr0NGjix5IrsEIHW3UcTKyzrOQOFv+jRZX+Tt2FLfzWTuay5LnYu595XoxSDgPcwe+9YXsIbvXSbYE3fzXSQs/dtB33mrc5kB6xzdFUHB94rUXPb46rRpmem4Oy2P1M3n2w/vY1dyN98XWQqwGmq6zYYX8062c1fBCfgvYTkBUWP0uopmVuRlEjMvazB55Ie4GRoKIdRBvTcGpP8T7biDO6Ftb0sbbOmkApfqvfo20pBX0DpD3PEgMDp2AUUnlmkSZzZzXUitQXkPRiV0hh+0gy/06EIH4vNiG5VAVsij12K0pE9GRpSPUpqnlujiZ5uxVlRpu7g7ZMF13DgbJ6u6dgFhJ1ZpCmb2Q06wRZVI6shYRAwyQ7APg5wKmWqT9nM4AWaPrdK0za3IR7AdSYKvK33vhPpkTl7Zjk6R3mGI28mnek74tPwBbJfy3+3WS1TDmdNQJ//wJnjSiiOmENG36xFeLz4+FVfSCkUAeEjrrRQWMwuyob6Q/qmDKC6lwMwatCW9ryjtxRwZDHIlUgZzMyIoQM0WjcZ+DNJSvezpWnAmTl5tdPgtcceqwYauPK7ars8JrJqvJ/oB/SC3wFC39J9vRhiw9+yW/yY0Lyv2Iosl+nSe6RbCEwPwpiKVnH8BL3Q8YT57VmWUB2xrOsSy/D6d7llz6FpXSlIk1oFSJYltrQt8Ee9G1uaBnpjvBV30gkNzhr8nE9jYeoBR4LEiBNrYJumPtSgdJ2nUlKbQjqvfD2Ho3cXTfQ24JMctp966dlX/wVQSwMEFAAAAAgASKkNXVne50KyAwAASQoAABIAAAB0ZXN0cy90ZXN0X2RhdGEucHmVVktvHCkQvvevQJwYqd2eiaWV19JcojwUaRXtIdqLNUIYaA9JNxCgPfZG+e9b0HQP89o4ljUeinp8VH1VZdVb4wL66o2uWmd6ZFnYduoBqfHibzhWVT5YpgXzCH6tmGUvQfpQjcbe8UawwCZrUiH44UwbrTjr1L+SctMNvfb16U3rWC9HuZBB8kAHrb4Pk0m+UZ6bJ+loDONlGKU+sIdOUs96C3+UyO6fwDHoyUmZ9kyrFuDC/aKqKiFbFNHnCHS3VXC0jIMPT0twgu5U2JohUK86qZNFp7wymizuUqyEHq0hM807iPYhHskPjP5iD7LDd+gev33/+dPHz3hTI4y+qB7iAtx085yEHzqzQ5/eJQnDm5+LwxSB79N0kfQ5ajLvJeQcYAUyazY53wu0XoPfEQ0E2wMoIm9KP+dqcOq3Lp126ctm/paC5uvkOiYxM6ZxTHnpyT+sG+R754yrUc8C367xnFuPc3IvsYgUwXOmNweF3TOCMgcEGXkCNKbeDA7qDGyyUvx/EWcImE5mrUpeU62Wq6vVm2vWcP8UcRwfb65Wq3zc1GdcObPLnpY1WtVombVy/VvlfABIJxQvS+8lN1pc1oJq2ReyOOBJctzI7wPrPBkdnLnXY/1JKuVNmdqpl6AWOjgGVPGyA8Z4ap3qmXuBbLuBhwHybp0En09KP8JzTSChtzQOmrs0X3Ly4w08YbpD1whzxYUw/s1y9SfIHAAJky8pRkqlcbMejcEiHnH8ksowqsxdX+jNsr2Xpv8mlCMQBxrcr7+4IWd31r2o0RqHlBbyGT6RY/pRktVtwV2SUF6jFuei/0jaP5v8KLxoghn4luRy8q3sGd0yvwXImD3wESXZvyQ/NQ41P/Qx3U2c4eBo52CI0SCfA4mSRgy99SWJSwpzM+gA3FvdAlWDCayLfPRRsgQ64gIJyIpTpmiNpOZGQGHXeAjt1S1enEGa5zB0NHUyLobXYw0sDBEOtpGWInaUjMNiRr6cJFHrfvNbuPLYH5/1akzGqUelIVPZfEJyU3Q3bAo7hGONPwqNi5mtL0ZKLyx3x37qlUO8cGAd7ErNNJ8nJmXWSuDeOLooLDmbOlSwl+jkZMDVp5Nqnk+Xkhy7N/ba8a6e277+VWfDILzN6y8PF3B3eZuTFLFGRZHks4VZJEX5Hj8S/YzSmUTfnNUry+qPKroP+YrSpgUZ/8UQ6U1jaU8W6V4/TEU/tjlgw6yFD5dIHulTMu+n8OkdeHO8pk/Ui+ilyT52Vf0HUEsDBBQAAAAIAEipDV2QRW/zLAYAAHsSAAAdAAAAdGVzdHMvdGVzdF9ncmFwaF9zZXF1ZW5jZXMucHmtWG2L4zYQ/p5foRoKdut187JdSqgLB8dBoZSD67cQjDZWsurasirJd5vb7n/vaCTbsvNy+dDsEq+lmUejmWdGo92rpia7Rh4Jr2WjDCkZk/Z9trczkpqnij92kx/hdTbzL6KtQY1qImQ3JKkoYQB+ZTlzCFrtspIa2kFoWsuKFbtGGH5om1YXkqp/WmaKPa/YoHNQVD4VmsGU2DHdqcczAp93FT8IVn6SFTcpjlA7UkjFpGpAXLOy0MPsY8ursgcrHLZhQjdKOwlNP7OrAp9hBdhHIFQx+kwPLJ0lg9WDAVwceq8FVqHJejablWxPtGFyZV2x54c4wBUH87QmXBiSk/sUxBQvWTewTMjdb6TkO7NGwxQzrRLkFV/sJ0LYaB0MueHxAiAwGUkvyLv1rTz+MRFTIMYVRlSzXWu4dWTTKlBUzRcNan+pdqpkqDpAyFVbWdyootoUhtfMmh5N7XBgTJSyARfAQlVbC6v2CWfI7x+nKiUAcUENb8Q5vffD9BnlXuGJ6qdCULBK0h3a2Qpu7gxoT3VqjhEfFpNNxXdHq1OqRhZfuCibL1MtR7MSvLeztqA0vrAyEH2buW/HGeqo79gd4zcGJiXg7YKXek0qrs0GjNim5KCaVsIoipB/yZ+NYMAh+0AahYnk6IQaINJpkkaRffSKC71lO/15vX6smt1zPo8c/SDGIC7L7D1k+QcF7opDLlo9S50hGf3WHTxM4Z/BlK8QOLfpl16/uv29RWQPJrkXSIlu29sAoegIaEuKAxptITon63BA1AMGMgPP0KLFPLM/3iDyPVmRH8niFsMmxBvQFgHa8ga0twQfL+B5ITOqqDiwGLI4tuFIyA9klZLSHCXLYXpfNdSslkmmmH6iMhBMycoBHT2QpkrRY7wJjLlgxgAPTHu4T8JaFLIKF0rJS0qOiaewTaACS7LG2rIzQ50v4ATx2VIAzWjlarGOTS0Lexqt8RBK1r6ui7KyhL5W4OPe/+PciYyiXEQp5kvsXPhLkoBT+plRgXZbdN8UajlUdrd+1q/7kqGDSZ6TGCJw3/v3gvxxIu+EwacGiXLInF6BUSlAi1axx/AV9Jiegm7WKblbbM+tzMoDJIQoWWjpMiWL5S1Lw2zJ9/s4BPOxkkaB4yzZ2qpyexmtb8lVnSCMUg9YQl+4zheJtWoxUre6lrWiKRnWZZR5uEd29oOWoB65H2N6bEcDYOIYu7SLrMagfjPYlW7Bby0lHWFHy/c0Jj8FB6xXzYT8GiUZAycYHV/WG613q9LE2poKvodIZ39rOHgC/SBFg+gcqITUhFMe/Fc8QhdwYGX8vybhZg4sTMkSs+bnlDxsk6Bweql+4FrnlK+6lgmolN6ctJv51nJqeU50RNPMNFgxkIKbzm6oiJvO8m3oxZ1qtHa7HXxjq5w7An0XWXBdsFqaY+dV7zdw63Ca+tP+Nj+f+tqdwWG5e8By58dv9KmvaINb7cdmDsLYrBmCGvmu2TY3tqzazinx7Yw7L7BBzi9317Hf1yiCTm0DjS41rY4wbpG0k2U0rRqA3GKZm6OR7tUeZB1G121AWTRMadeKAWiGopASJ4XoW5Bdb3MFMeAH+qrASNUMOkcqwY9wEDatgaeCTNXPUGXNEwwUijWqZArqc8cT3399M6tWWzj/QRaV+usIaJ7eTgYOOdte8rA12EDhtGRfuscKHmGqDhF3eo7UsbUjCcRw61cFcOnjaGmQsWIXljsOaPOT9mSy8q2yaAQXfSBG5uC9hnyglYaHe7HfoXk1M9TefvPXt1ElctHq4nbm3urbpn60b0ombEQcDOzmpJXdjivV3PPghoO+w32xrQSGGYIO6vbLBnxE4O7+PtwXMaVU4bpBDU9o44ILv2rFhX4OD6ucjM4t3FLmF3Hpfemycabv33QAtu3fQmO8mM/nV9v/oZ+2ouEh9Ad9ZBVivptAvSXg6s4RsTXdXcdcJdD8K8uXIEuw88qRMV0jYV0FW77+b5EhIx22k4ZW6QFANWNlfg+hrbngdVtb/+LVG6ZhEHoqHOzM0fn96ZloeysH6pqvh7k72iF0z7a8QGLEXuAcz7CnSzIwtRI0hgbuO9vAZbqtJ+XTA/6ak9XsP1BLAwQUAAAACABIqQ1dVvOhJyUCAAB/BAAAEwAAAHRlc3RzL3Rlc3RfbW9kZWwucHl9VMGO2jAQvfMV1p4cKesusN1DpfTQHri0e2lvCFlDMglWHTu1He3Sr+/YDoFQqREiZPz85s3zC6ofrAvMjP1wZuCZGVYql4J19Wm1ap3tmXe1qK1pVcemVW2hkbl0hfS2QX1B7L5++/Hz++5kfXjFULKdg+H0BUIkXTXYsoA+yLRFtta9gWskmEYeof4VH3jBHj+zV2vw04rRNfWvblvzh3z3H47gUZyh1w8lm4uDgzqoGrSMy1qZCVIkwqy2uhPKW4QwOqQOownVS8lqDd5Pj9ty0lHlW2bCpkNPTGEcNPJknAhovHV8v38q2bpk9L05lGxPPzfpsz4cCkZzM8mUYQ5Mh3xTZL5jdCkqmy3jqR4vj79HNDXK9yo3iu7LdH7cDIKIGtsL8hdGHaQzHd8WwljXg+Ze/cGKU/Pnkr0URSFa8jLwopzZA7gOgzxXyyGS+htYHFgq0+B7lWa/rhgyNXvl7zjIuu2FI4+pbaeCLxkEwgRlDY2cDoUnAzKIvMeUt4gV/gQDsqpicYrtAjGzLEHPC1CWBFrX2nrkN3vGnjeqr9ZFOYEoeJ5OJMqzulrj48elIDQ8iRUUjyD9AEFR0GbGIgrYLARqze/6v6HqTnEs6j43ngxbi6d/m99cMTvT/pig/2pJezO7MaIdTR3LoEXtLGWbUM5Sfi4HkuwXlywU4vpKLuYxZz6Agx4DOtE5aJiiPxAb0kub9M3LV4VzyXNK+19QSwMEFAAAAAgASKkNXav6RP/+BAAANQ4AABsAAAB0ZXN0cy90ZXN0X3ByZXByb2Nlc3NpbmcucHnVVl1v2zYUffevIAQMkDZFldR2aA24wLauQ4oCCdruyTAIRrqy2UokQdJJvCL77bskJVtK1DSvUwzH4v0+9/CSjZYdqaQ6EN4pqS2pAZR7XzROopjdtfxqEF7i62LRv4h9h2bMEKGGJcVEjQv4UfUieDC6yiopGr4dnLSS1TQsnVSUBqVlBcZwcdT8i3Fx3qm9BZ2SD8C+si18Yg1cHpWlXiwWlx8v3v/5x2f68eLiM1n5JGNKG94CpUmmwcj2GuIkU0yDsGZdbNCohoYozSrLK9b26cTJckHw6fNdjVONvcQ9k3DPSBTkJnK/r5iB7MC6NkqfpH/KwFm2XEysk1E262gCUbRZRxwLY5ZLQRuJVVq3BoJdtVBHG8z+HWsNeBca7F6L3lNfvEULymtEhDccNJbZ7jthKILU/6aCdUCN1RjPPMDmIXhezowBbN2QdM0s87k+CISrOWa5ItHfwgWqlySPxi5Y28YcazWWiQriYJYSzCchWDAJC4SLJwVLjj0foUgbjZHjhJy9QcZmb9H+nVsJpWp5Y7DQ9ca/uZBGtdymGG8v8J9sGgPWJRDHkdVI1iglZZ6SPElJHF2zlte+P7j8MiVFHtYd8GGlxJUe1iECFzXcOpeaia0rGiONVNyDfveAeTXIThsHg1/6ZJKJpisgY0qBqONvE4l7Il9NtOyreij/wK6gRXn0W0R406f2Eyld03ICSC4S/R7NGDYFWvk0Z4W03mPEill4TK1EoVAZF40L7nP0bAlAE5w0fUanbAIyP5Nyxh9yxBHJ1Vtk+YzCu1bekPO3KG8ihPbm7JuPeXf2zYe5myv0k9zrCsj5pUOpyDP3V8wpvsWec+HJMNUu57Q/8w71WaecYl48K8pnZV68Inm+9J85m9EmWgZgZpQorZjCUQC0Zofg/KyYTYFS42sLY5TXHpYekawy14/aIO+CyffyMFjbA7fLR4DearlX9/Xv43CXjIfdeD/Hbick48F3bwxwa6gU7YF6dlEkl9PAMXQNhp42MvW2zllsO0Xd6bj0580Th+Px1EKV75xncTDAOQdQr16UQ01m31rvuNfLMGeXrTA4NjrqITHx7HRLh608mdDBZRYKvs3MjinoJ3KZzyiOUJhqv5zz6nB6RG06Bfy8C4YdWObGOA5zLXFy1aPjIjpahDZFm4nL4w5/qrvBYNbbMA6IkHbWYQPM7ySpa9BT4zCZT60ySATQmbOjHbul6x85y/xOiN0gTTaJQ694nf2wKegaj7I3br5N2ZYZhheggbHp/VomtDiqhevJiE7ZF4NHWZLBLTdItadYYegv8govkGOz0S7cOu6Zr1wZesPtTu4t7Xjgrp/lx1tHeEP+44nAwtFYvkhJbQ8KVrjmEX9e+tue41z8KiXPQ4Y8XCDRdnSdjEHJamdWRUqumK121PB/YIUedxz5oJFiqzx7neIdRO3YqsAzvQWmhUusF+Z58XBOnZ4dr/ESQjvEmSNpQa/cqTPd1MfdCzVm1+c53ddxKHyC9aDocHOMFnIAzWshGtafNtssGFC8SFWtNMiAU8CUDJ7vtyO4N/ca4ftDNeBE24IABAFH1WxzNDvE6yMy69yV7WtHqWBic0JtPVrGdmX5WFY62XP39WIqGNbwBnUUbGao8OPml5Pml/+L5rtp5G5FBjEb9zPJmDj0e/JpDFj/e/TTx9kMpJgVaSvbVQFnvyaL/wBQSwMEFAAAAAgASKkNXQR21b3eAAAArwEAAB0AAAB0ZXN0cy90ZXN0X3Jlc3VtZV9jb250cmFjdC5weW2QwWrDMAyG73kKkZMDWaCDHRbIHmHsDYySKK0hsT1ZhcHou0+119Fs88GS+D9Zv7xw2CCinFY3gttiYIE3Lavqu5DAk1bLlUs8dcLovPPHG2yZkjJk2R9tEhRqYZdrHFeyJ0z6TDXTAqIdGeFw9rMVdtGiJnekkS3aq60+u2ng4QVeg6e+Aj3ZU7ehP+NqE9FsDs9NVvJQGO4cmCLQR6RJaFatdLNONE9F/LuDyXdRcRKd828jpkS3P+roXTFT6PZn4A7crfhZj3UPjy3UqPFwaWAYfhNZUSKTl6b6AlBLAwQUAAAACABIqQ1d7r+nVZICAACaBgAAFAAAAHRlc3RzL3Rlc3Rfc3BsaXRzLnB5lVTbitswEH33VwyCgk1Tr9Nd0jbgQkvpU/9gCUKxx1mBLauSvBdC/r0jS7Gdkt1SkYdYOnPmdmZkp3vjQAtVCwv003WSNKbvwJoqt7qVzoIMIGGtPCh+MP2geXhawaNoZS0chgsulcODke4lSZIaG7Avyj2gk1Uww5o3RnSYZvDhK/nKfwgnfvqbbQJ0TP9koYT73fjV9AZsP5gKibfGZ5AKjFAHTDdZwPsTEGTVsEpoNxi8iUbHpfEpr+wjm6w8N3njsl6wFgvaczi50BpVnR4vXvxhnEcHjWzJS822MZjVG9jglKDhzxWoFZ0+8zUsJnHaHoPBiV0x+SX22BKcfWMgG0hjZjc3sC7g/UURM3gHH6EsoQBsLQL7/hfhKQutQCqluuiRp7VZ7KxD60JTLRcGeVAHNXiPVNuoB8ufpHvohyAMY7FysldpLPMoBWrcqyKJgdihdQS7or907qe3mBNpfUF41bdDp8pYnvmVAkLDnRFSeVdjUGWRf5oRUdZ0PwF43/CFIeHXMz4E5etTrov52iLW5d3H+aKiQQvzIpzDTjtb3sbnkO00QJTwa7OVhpLkY87BjGqDNKIT5J5ZJ9xg2c63mmn/XrMl1KK74CEL74XtMm9xZGOSbAVsLoX/8m1np5FIUynGvCnUJVM+Xu5fUhaqQkLOJvZcDUr+Hs7NneNOJ7q8E89pNoaxXoKiD4N+G90zbBovp0ecOhSznSc1JrGFIt/czk1YpuTfis2Xy7OAjvl60G1xee4C6LQch33vHniIj0S8VJmtUAkj+zArdtAB83+T4HfWOVe/tVIvWQrt82JtvTktYUjO62I1kXmS9Yp2xQqIcL3JJrrrpb82P6H2569/EEw6ZbtXlJr8AVBLAQIUABQAAAAIAEipDV3J/i8VXQ0AAHAfAAAJAAAAAAAAAAAAAACAAQAAAABSRUFETUUubWRQSwECFAAUAAAACABIqQ1dKIu3I0QAAABJAAAACAAAAAAAAAAAAAAAgAGEDQAAdHJhaW4ucHlQSwECFAAUAAAACABIqQ1dgRDlOKIFAAA1DgAAEAAAAAAAAAAAAAAAgAHuDQAAYXNzdW1wdGlvbnMueWFtbFBLAQIUABQAAAAIAEipDV3T/+qc2QQAAOkJAAASAAAAAAAAAAAAAACAAb4TAABwYXBlcl9hbGlnbm1lbnQubWRQSwECFAAUAAAACABIqQ1dwWaIt08AAABVAAAAEAAAAAAAAAAAAAAAgAHHGAAAcmVxdWlyZW1lbnRzLnR4dFBLAQIUABQAAAAIAEipDV2/f9bhXwQAAEQMAAAXAAAAAAAAAAAAAACAAUQZAAB0cmFjZWFiaWxpdHlfbWF0cml4LmNzdlBLAQIUABQAAAAIAEipDV3Ah/q13QQAAFkKAAARAAAAAAAAAAAAAACAAdgdAABjb25maWdzL2Jhc2UueWFtbFBLAQIUABQAAAAIAEipDV2IAb2B1QAAAIcBAAAbAAAAAAAAAAAAAACAAeQiAABjb25maWdzL3BhcGVyX2ZhaXRoZnVsLnlhbWxQSwECFAAUAAAACABIqQ1dNJUNTrEAAABJAQAAHwAAAAAAAAAAAAAAgAHyIwAAY29uZmlncy9wcmFjdGljYWxfYmFzZWxpbmUueWFtbFBLAQIUABQAAAAIAEipDV3q/7diRQAAAEUAAAAPAAAAAAAAAAAAAACAAeAkAABzcmMvX19pbml0X18ucHlQSwECFAAUAAAACABIqQ1d07/U8IIDAAClCAAAEAAAAAAAAAAAAAAAgAFSJQAAc3JjL2JlbmNobWFyay5weVBLAQIUABQAAAAIAEipDV3NwHdfhwYAALQTAAANAAAAAAAAAAAAAACAAQIpAABzcmMvY29uZmlnLnB5UEsBAhQAFAAAAAgASKkNXbJmmGeyEgAAJEgAAAsAAAAAAAAAAAAAAIABtC8AAHNyYy9kYXRhLnB5UEsBAhQAFAAAAAgASKkNXbuzngNJBQAAZA8AABUAAAAAAAAAAAAAAIABj0IAAHNyYy9leHBsYWluYWJpbGl0eS5weVBLAQIUABQAAAAIAEipDV1iW+oAiA0AAAwyAAAWAAAAAAAAAAAAAACAAQtIAABzcmMvZ3JhcGhfc2VxdWVuY2VzLnB5UEsBAhQAFAAAAAgASKkNXYt1OC8WAwAAjAcAABIAAAAAAAAAAAAAAIABx1UAAHNyYy9tYWtlX3JlcG9ydC5weVBLAQIUABQAAAAIAEipDV12gFev0gcAAH0eAAAMAAAAAAAAAAAAAACAAQ1ZAABzcmMvbW9kZWwucHlQSwECFAAUAAAACABIqQ1dFsEFAuERAAAoSAAAFAAAAAAAAAAAAAAAgAEJYQAAc3JjL3ByZXByb2Nlc3NpbmcucHlQSwECFAAUAAAACABIqQ1dtpQP8FgKAABUIgAADQAAAAAAAAAAAAAAgAEccwAAc3JjL3NwbGl0cy5weVBLAQIUABQAAAAIAEipDV1UECZANQUAAM4QAAASAAAAAAAAAAAAAACAAZ99AABzcmMvc3RlcDJfc21va2UucHlQSwECFAAUAAAACABIqQ1dHP6O5kYHAAC4FwAAEgAAAAAAAAAAAAAAgAEEgwAAc3JjL3N0ZXAzX3Ntb2tlLnB5UEsBAhQAFAAAAAgASKkNXc7edst0CAAA8xsAABIAAAAAAAAAAAAAAIABeooAAHNyYy9zdGVwNF90cmFpbi5weVBLAQIUABQAAAAIAEipDV1HxCCgXQgAAE4bAAAZAAAAAAAAAAAAAACAAR6TAABzcmMvc3RlcDVfcmVzdW1lX3Ntb2tlLnB5UEsBAhQAFAAAAAgASKkNXasJBWP7BQAAMBAAABsAAAAAAAAAAAAAAIABspsAAHNyYy9zdGVwNl9hcnRpZmFjdF9zbW9rZS5weVBLAQIUABQAAAAIAEipDV3JxmLs3RgAAN9eAAAPAAAAAAAAAAAAAACAAeahAABzcmMvdHJhaW5pbmcucHlQSwECFAAUAAAACABIqQ1dGZ3ioR4NAABQKgAACgAAAAAAAAAAAAAAgAHwugAAc3JjL3Zpei5weVBLAQIUABQAAAAIAEipDV1Z3udCsgMAAEkKAAASAAAAAAAAAAAAAACAATbIAAB0ZXN0cy90ZXN0X2RhdGEucHlQSwECFAAUAAAACABIqQ1dkEVv8ywGAAB7EgAAHQAAAAAAAAAAAAAAgAEYzAAAdGVzdHMvdGVzdF9ncmFwaF9zZXF1ZW5jZXMucHlQSwECFAAUAAAACABIqQ1dVvOhJyUCAAB/BAAAEwAAAAAAAAAAAAAAgAF/0gAAdGVzdHMvdGVzdF9tb2RlbC5weVBLAQIUABQAAAAIAEipDV2r+kT//gQAADUOAAAbAAAAAAAAAAAAAACAAdXUAAB0ZXN0cy90ZXN0X3ByZXByb2Nlc3NpbmcucHlQSwECFAAUAAAACABIqQ1dBHbVvd4AAACvAQAAHQAAAAAAAAAAAAAAgAEM2gAAdGVzdHMvdGVzdF9yZXN1bWVfY29udHJhY3QucHlQSwECFAAUAAAACABIqQ1d7r+nVZICAACaBgAAFAAAAAAAAAAAAAAAgAEl2wAAdGVzdHMvdGVzdF9zcGxpdHMucHlQSwUGAAAAACAAIAAXCAAA6d0AAAAA"

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as project_zip:
    project_zip.extractall(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

mounted_data_dir = next((path for path in MOUNTED_DATA_CANDIDATES if path.exists()), None)
if mounted_data_dir is None and next(Path("/kaggle/input").rglob("dataset_summary.json"), None):
    # Kaggle may choose a normalized mount slug that differs from the API slug.
    # The data loader recursively selects the validated manifests below this root.
    mounted_data_dir = Path("/kaggle/input")
if mounted_data_dir is not None:
    DATA_DIR = mounted_data_dir
else:
    mounted_entries = sorted(str(path) for path in Path("/kaggle/input").iterdir())
    raise FileNotFoundError(
        "Dataset input is not attached. In the Kaggle editor choose Add Input -> "
        "dungnguyen28101991/cicddos2019-parquet, then Save Version / Run All. "
        f"Current /kaggle/input entries: {mounted_entries}"
    )
print(f"Step 7 sampled end-to-end CPU project ready; using dataset at {DATA_DIR}")


In [ ]:
command = [
    sys.executable, "-m", "src.step6_artifact_smoke",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--samples-per-file", "2048",
    "--sequence-length", "16",
    "--sequence-stride", "8",
    "--batch-size", "64",
    "--device", "cpu",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
import json

summary_path = OUTPUT_DIR / "step6_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert summary["device"] == "cpu", summary
assert summary["sequence_leakage_status"] == "passed", summary
assert summary["expected_not_yet_run"] == ["ablation_comparison", "cfaco_convergence"], summary
assert len(summary["report"]["produced"]) == 11, summary
assert (OUTPUT_DIR / "report" / "report_status.json").exists()
assert (OUTPUT_DIR / "artifacts" / "benchmark.json").exists()
step7_acceptance = {{
    "status": "passed",
    "step": 7,
    "mode": "sampled_end_to_end",
    "device": "cpu",
    "source_summary": str(summary_path),
    "produced_report_groups": len(summary["report"]["produced"]),
}}
(OUTPUT_DIR / "step7_acceptance.json").write_text(
    json.dumps(step7_acceptance, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
step7_acceptance
